# ESM-DMS real-data analysis (local)

Local variant of `real_data_esm_analysis.ipynb`. Runs the full workflow for the `TpoR` dataset using `esm2_t6_8M_UR50D` (6 transformer layers + embedding = 7 total).

Differences from the cluster notebook:
- Single dataset (`TpoR`) and smallest ESM2 model (`esm2_t6_8M_UR50D`).
- Embedding and inference run locally via `embed_all_sequences()` and `run_feature_inference()` — no Slurm jobs.
- `local_or_disk="both"` caches embeddings and results to disk so re-runs skip recomputation.

In [ ]:
from pathlib import Path

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from esmDMS import CellularDMSInput, ESMDMSConfig, esmDMS

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break

DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_data"
ANALYSIS_DIR = DATA_DIR / "esm_data_analysis"
SEQUENCE_DIR = ANALYSIS_DIR / "sequence_data"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"

SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="darkgrid")
REPO_ROOT

## Dataset Registry

Single cellular dataset (`TpoR`) using a MaveDB nucleotide-count file.

In [ ]:
DATASETS = {
    "TpoR": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "TpoR_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "TpoR_nucleotide_counts.csv",
    ),
}

pd.DataFrame(
    {"dataset": dataset, "kind": "cellular", "save_dir": str(SEQUENCE_DIR / dataset)}
    for dataset in DATASETS
)

## Controls

`esm2_t6_8M_UR50D` has 6 transformer layers plus the initial embedding layer, giving 7 total hidden states (indices 0–6).

In [ ]:
# TODO(esmDMS.py): add a public available_layers() method so notebooks do not hard-code model layers.
LAYERS = list(range(7))
REPRESENTATIVE_LAYER = 6

ABSTRACTION_METHOD = "Embeddings"
ABSTRACTION_PARAMS = {"norm_scheme": "none"}
NORM_SCHEME = ABSTRACTION_PARAMS["norm_scheme"]

## Create esmDMS Runner

Each dataset gets its own `esmDMS` object with a dataset-specific save directory.

In [ ]:
runners = {}

for dataset, input_data in DATASETS.items():
    config = ESMDMSConfig(
        embedding_model="esm2_t6_8M_UR50D",
        embedding_method="per_residue",
        local_or_disk="both",
        save_dir=str(SEQUENCE_DIR / dataset / "local_t6"),
        dataset_name=dataset,
    )
    runners[dataset] = esmDMS(input_data=input_data, config=config)

runners

## Process Raw Data

Raw input parsing is handled by `esmDMS.process_raw_data()`.

In [ ]:
processing_rows = []

for dataset, runner in runners.items():
    runner.process_raw_data(drop_stop_codons=True)
    df = runner.sequence_dataframe
    processing_rows.append({
        "dataset": dataset,
        "kind": "cellular",
        "rows": len(df),
        "sequence_count": df["SequenceIndex"].nunique(),
        "replicate_count": df["Replicate"].nunique(),
        "generation_count": df["Generation"].nunique(),
    })

processing_summary = pd.DataFrame(processing_rows)
processing_summary.to_csv(TABLE_DIR / "raw_processing_summary_local.csv", index=False)
processing_summary

## Current Class Cache Status

This checks the paths that the current `esmDMS` class will use.

In [ ]:
# TODO(esmDMS.py): add a public cache_status(layers, abstraction_method, norm_scheme) method.
cache_rows = []

for dataset, runner in runners.items():
    for layer in LAYERS:
        cache_rows.append({
            "dataset": dataset,
            "layer": layer,
            "embedding_path": str(runner._embedding_path(layer)),
            "embedding_exists": runner._embedding_path(layer).exists(),
            "inference_path": str(runner._inference_path(ABSTRACTION_METHOD, layer, NORM_SCHEME)),
            "inference_exists": runner._inference_path(ABSTRACTION_METHOD, layer, NORM_SCHEME).exists(),
        })

cache_status = pd.DataFrame(cache_rows)
cache_status.to_csv(TABLE_DIR / "current_class_cache_status_local.csv", index=False)
cache_status

## Embed Sequences

Runs embedding locally via `embed_all_sequences()`. This replaces the Slurm embedding batch jobs and merge step from the cluster notebook. With `local_or_disk="both"`, derived per-residue and mean-pool caches are written to disk and skipped on re-runs.

> **Note:** `create_embedding_batch_job()`, `create_embedding_batch_merge_job()`, and the merge polling loop from the cluster notebook require Slurm and cannot run locally.

In [ ]:
for dataset, runner in runners.items():
    print(f"Embedding {dataset}...")
    runner.embed_all_sequences(layer="all")
    print(f"  Done.")

## Run Inference

Runs inference locally via `run_feature_inference()` for each layer. This replaces `create_inference_job()` and the Slurm inference array from the cluster notebook.

> **Note:** `create_inference_job()` and `run_inference_job()` from the cluster notebook require Slurm and cannot run locally.

In [ ]:
for dataset, runner in runners.items():
    for layer in LAYERS:
        runner.run_feature_inference(
            layer=layer,
            abstraction_method=ABSTRACTION_METHOD,
            abstraction_params=ABSTRACTION_PARAMS,
        )

## Load Completed Inference Results

Results are already cached in memory from the cell above. This cell re-loads them from disk to confirm the disk cache is consistent.

In [ ]:
inference_results = {}

for dataset, runner in runners.items():
    inference_results[dataset] = {}
    for layer in LAYERS:
        inference_results[dataset][layer] = runner.load_inference_results(
            layer=layer,
            abstraction_method=ABSTRACTION_METHOD,
            norm_scheme=NORM_SCHEME,
        )

## Inference Summary

Run this after completed inference results have been loaded from the class inference cache.

In [ ]:
# TODO(esmDMS.py): add an inference_summary() method that returns this table from stored results.
inference_rows = []

for dataset, layer_results in inference_results.items():
    for layer, result in layer_results.items():
        inference_rows.append({
            "dataset": dataset,
            "kind": "cellular",
            "layer": layer,
            "n_replicates": result.s.shape[0],
            "n_dimensions": result.s.shape[1],
            "gamma_opt": result.gamma_opt,
            "s_joint_mean": result.s_joint.mean(),
            "s_joint_std": result.s_joint.std(),
        })

inference_summary = pd.DataFrame(inference_rows)
inference_summary.to_csv(TABLE_DIR / "inference_result_summary_local.csv", index=False)
inference_summary

## Replicate Consistency By Layer

Layer-wise replicate consistency plots are delegated to `esmDMS.plot_avg_rep_correlations_by_layer()`.

In [ ]:
for dataset, runner in runners.items():
    runner.plot_avg_rep_correlations_by_layer(
        layers=LAYERS,
        abstraction_method=ABSTRACTION_METHOD,
        norm_scheme=NORM_SCHEME,
        comparison="selection",
        label=dataset,
        output_path=FIGURE_DIR / f"{dataset}_t6_selection_replicate_correlations_by_layer.png",
    )
    runner.plot_avg_rep_correlations_by_layer(
        layers=LAYERS,
        abstraction_method=ABSTRACTION_METHOD,
        norm_scheme=NORM_SCHEME,
        comparison="fitness",
        label=dataset,
        output_path=FIGURE_DIR / f"{dataset}_t6_fitness_replicate_correlations_by_layer.png",
    )
    plt.close("all")

## Representative Replicate Scatter Plots

Replicate scatter plots are delegated to `esmDMS.plot_rep_sel_comps()` and `esmDMS.plot_rep_fit_comps()`.

In [ ]:
for dataset, runner in runners.items():
    runner.plot_rep_sel_comps(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        norm_scheme=NORM_SCHEME,
        label=dataset,
        output_path=FIGURE_DIR / f"{dataset}_t6_layer{REPRESENTATIVE_LAYER}_selection_replicate_scatter.png",
    )
    runner.plot_rep_fit_comps(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        norm_scheme=NORM_SCHEME,
        label=dataset,
        output_path=FIGURE_DIR / f"{dataset}_t6_layer{REPRESENTATIVE_LAYER}_fitness_replicate_scatter.png",
    )
    plt.close("all")

## Shuffled-Frequency Control

This notebook does not implement shuffled controls locally.

In [ ]:
# TODO(esmDMS.py): add a class method for shuffled-frequency controls that shuffles
# within each (Replicate, Generation), reruns inference, and returns InferenceResult
# objects compatible with the plotting methods above.